In [10]:
import os
import json
import re
from openai import OpenAI

# 第一步：先对文本切块，将文章切为每一小段进行处理

In [11]:
# 定义一个切块函数
def chunk_document(file_path: str) -> list[str]:
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 基于正则分割文档，将长文档切割成块
    chunks = re.split(r'(?=\n\d+\.\s)', content)
    # 去除收尾空格、换行、制表符，只保留有效长度>50 字符的文本块
    cleaned_chunks = [chunk.strip() for chunk in chunks if len(chunk.strip()) > 50]
    
    print(f"[Step 1] 文档切分完成，共切分出 {len(cleaned_chunks)} 个文本块。")
    return cleaned_chunks

file_path = "document.md"
result_chunks = chunk_document(file_path)
result_chunks[0]

[Step 1] 文档切分完成，共切分出 4 个文本块。


'1. 全球的“双碳”目标。\n\n   探讨任何能源技术的变革，都绕不开当前的全球大背景——气候变化。为了避免极端气候的不可逆破坏，全球形成了一个硬性共识：必须减少温室气体排放。目前，欧盟、美国、日本等发达经济体普遍承诺在2050年实现碳中和。在这个宏大的目标下，能源行业的脱碳是重中之重，因为全球温室气体排放中，能源生产和消费占据了绝对的大头（约占70%以上）。这不仅是一场环保运动，更是一场重塑全球工业格局的能源革命。\n\n   **核心内容：**\n\n   **共同危机：** 应对全球气候变化，《巴黎协定》控温目标（1.5℃/2.0℃）。\n\n   **全球共识：** “碳达峰”（Carbon Peak）与“碳中和”（Carbon Neutrality）成为全球行动纲领。\n\n   **时间表：** 超过130个国家和地区提出了碳中和目标（多数定在2050年）。'

# 第二步：安全配置 deepseek API 方便后续调用

In [12]:
from dotenv import load_dotenv

# 1. 自动寻找当前目录下的 .env 文件，并将其内容加载到系统环境变量中
load_dotenv("deepseek_key.env")

# 2. 安全地获取 API Key
api_key = os.getenv("DEEPSEEK_API_KEY")

# 3. 做一个安全校验，防止没配好 Key 就直接往下跑报错
if not api_key:
    raise ValueError("⚠️ 未找到 API Key！请检查是否创建了 .env 文件并在其中配置了 DEEPSEEK_API_KEY。")


# 4. 初始化客户端
client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
) 

# 第三步：调用 Deepseek ，设计 Prompt，批量生成问答数据

因为 DeepSeek 主要依靠 Prompt 和 response_format 来约束输出，

我们需要在 Prompt 中非常清晰地给出期待的 JSON 结构示例。

In [9]:
# 定义生成问答的函数（调用 Deepseek API）
def generate_qa_from_chunk(chunk_text: str) -> list[dict]:
    """
    调用 DeepSeek API 并强制输出 JSON 格式。
    """
    # 因为 DeepSeek 主要依靠 Prompt 和 response_format 来约束输出，
    # 我们需要在 Prompt 中非常清晰地给出期待的 JSON 结构示例。
    prompt = f"""
    请阅读下面这段【文章片段】，并根据该片段，为大语言模型的微调生成 3 个不同类型的训练数据。
    必须包含以下三种类型：
    1. 概念解释：针对片段中的专业术语进行提问（此时 input 为空）。
    2. 内容总结：要求总结片段的核心意思（此时 input 为该片段）。
    3. 逻辑推理：基于片段内容提问一个为什么（此时 input 为空）。
    
    【重要指令】
    请务必输出一个合法的 JSON 对象，包含一个名为 "items" 的数组，数组内部是包含 "instruction", "input", "output" 三个字段的字典。示例格式如下：
    {{
        "items": [
            {{"instruction": "什么是碳中和？", "input": "", "output": "..."}},
            {{"instruction": "总结这段话的意思。", "input": "...", "output": "..."}}
        ]
    }}
    
    【文章片段】：
    {chunk_text}
    """
    
    try:
        # 调用 DeepSeek 的对话 API
        response = client.chat.completions.create(
            model="deepseek-chat",  # 推荐使用通用对话模型，性价比极高
            messages=[
                {"role": "system", "content": "你是一个专业的数据标注专家，请严格按照用户要求的 JSON 格式输出数据。"},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"}, # 开启 JSON 模式，确保返回合法 JSON
            temperature=0.7 
        )
        
        # 解析 DeepSeek 返回的文本内容
        result_text = response.choices[0].message.content
        result_json = json.loads(result_text)
        
        # 提取 items 数组
        return result_json.get("items", [])
        
    except Exception as e:
        print(f"调用 DeepSeek 生成数据时出错: {e}")
        return []

# 第四步：收集数据与格式化


In [ ]:
# 定义主函数调用前面定义的方法处理数据
def main():
    input_file = 'document.md'
    output_file = 'new_dataset.json'
    
    final_dataset = []
    
    # 执行第一步，先切块
    chunks = chunk_document(input_file)
    
    # 循环处理每一个块的内容
    print("[Step 2 & 3] 开始调用大模型生成问答对...")
    for i, chunk in enumerate(chunks):
        print(f"正在处理第 {i+1}/{len(chunks)} 块文本...")
        qa_pairs = generate_qa_from_chunk(chunk)
        final_dataset.extend(qa_pairs)
    
    # 收集数据并处理
    print(f"[Step 4] 收集到 {len(final_dataset)} 条数据，准备保存。")
    with open(output_file, 'w', encoding='utf-8') as f:
        # ensure_ascii=False 保证输出的 JSON 中文正常显示而不是 Unicode 编码
        # indent=4 让输出的 JSON 具备良好的可读性（缩进空格数）
        json.dump(final_dataset, f, ensure_ascii=False, indent=4)
        
    print(f"🎉 数据集已成功保存至 {output_file} ！你可以直接用它进行微调了。")

if __name__ == "__main__":
    main()

[Step 1] 文档切分完成，共切分出 4 个文本块。
[Step 2 & 3] 开始调用大模型生成问答对...
正在处理第 1/4 块文本...
正在处理第 2/4 块文本...
正在处理第 3/4 块文本...
正在处理第 4/4 块文本...
